# 07 - Checks of the generated pairs (CPU only, no GPU)

Runs on a laptop in a few minutes, or on Kaggle with the accelerator set to **None** (CPU sessions do not use GPU quota). It trains no BERT model; it uses simple word-count models (scikit-learn) to answer four questions about the pair dataset that a reviewer will ask:

1. **Pair sets.** The same checks the GPU notebook runs before training: hashes, one sarcastic and one serious row per seed, no leak into dev/test, and the sizes of the four pair sets. Running it here first means a broken file is found without spending GPU time.
2. **Style probe.** Can a word model tell generated headlines from real ones? (High = visible "LLM style".)
3. **Real-trained reader.** A word model trained on the real labelled headlines reads each generated side. Do the sarcastic sides look more sarcastic than the serious sides? This is evidence independent of any LLM judge.
4. **Shortcuts.** Do the outlet cue words, or headline length, predict the label in the pairs as they do (or do not) in the real headlines?

**Inputs.** Locally: the processed data in `data/processed/`, plus the four pair files anywhere inside the repo (for example `results/pairgen/` and `results/judge_v2/` after you file the zips). On Kaggle: add `gainsplit-processed` (new version with the pair files) as input.

## 1. Get the code

In [ ]:
import os, subprocess, sys
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    REPO_DIR = "/tmp/GainSplit"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Ariffurrhmn/GainSplit.git", REPO_DIR], check=True)
else:
    REPO_DIR = str(Path.cwd().parent) if Path.cwd().name == "experiments" else str(Path.cwd())
sys.path.insert(0, f"{REPO_DIR}/src")
import pandas as pd
pd.set_option("display.width", 200)

## 2. Find the data
`DATA_DIR` is the processed folder. The pair files are searched for under `PAIRS_ROOT` (the whole repo locally, `/kaggle/input` on Kaggle); if a file is found twice, both copies must be identical.

In [ ]:
from gainsplit.data import find_data_dir, verify
if ON_KAGGLE:
    DATA_DIR = find_data_dir("/kaggle/input")
    PAIRS_ROOT = "/kaggle/input"
    OUT_DIR = Path("/kaggle/working/pair_checks")
else:
    DATA_DIR = Path(REPO_DIR) / "data" / "processed"
    PAIRS_ROOT = REPO_DIR
    OUT_DIR = Path(REPO_DIR) / "results" / "pair_checks"
os.environ["GAINSPLIT_INPUT"] = str(PAIRS_ROOT)
OUT_DIR.mkdir(parents=True, exist_ok=True)
verify(DATA_DIR)
read = lambda n: pd.read_csv(DATA_DIR / n, keep_default_na=False)
train, dev = read("headlines_train.csv"), read("headlines_dev.csv")
print("data:", DATA_DIR, "| train", len(train), "| dev", len(dev))

## 3. Pair sets (the GPU notebook's pre-flight, on the CPU)
Expected: `unfiltered` 44,834 rows (22,417 pairs); `filtered` and `matched` 34,630 rows each, with the same pairs per model (Gemma 7,533, Mistral 4,986, Qwen 4,796); `unpaired` 22,417 rows (one side of each pair). An error here means a file is missing or wrong: fix it before the GPU run.

In [ ]:
from gainsplit.data import load_pairs, load_cue_words, preflight
from gainsplit.plans import pass4
display(preflight(pass4(), DATA_DIR))
pairs, info = load_pairs("unfiltered", DATA_DIR)
cue_words = load_cue_words(DATA_DIR)
print(info)

## 4. Style probe: generated vs real
ROC-AUC of a word model (TF-IDF of words and word pairs + logistic regression, 5-fold cross-validation) that separates generated from real training headlines; equal numbers of each. 0.5 = cannot tell them apart, 1.0 = always can. A high value is expected and is not itself a problem: within a pair both sides are generated, so "generated" cannot predict the label. It is reported so the paper can state how visible LLM style is.

In [ ]:
from gainsplit import pair_checks as pc
style = pc.style_probe(train, pairs)
style.to_csv(OUT_DIR / "style_probe.csv", index=False)
style.round(4)

## 5. Real-trained reader
A word model trained on the **real** labelled training headlines scores every generated headline. Columns: AUC with the intended label as truth (real dev headlines as a reference row), mean P(sarcastic) on each side, and the share of pairs whose sarcastic side scores above its serious side (0.5 = no difference). This does not use any LLM judge.

In [ ]:
reader, reader_pairs = pc.real_reader(train, dev, pairs)
reader.to_csv(OUT_DIR / "real_reader.csv", index=False)
reader.round(4)

## 6. Shortcuts: outlet cue words and length
**Cue words:** AUC of a model that sees only the 200 outlet cue words (present / absent), in real training headlines vs in the pairs, and the share of headlines with at least one cue word per class. The pairs are meant to break the outlet-word link, so the AUC should drop towards 0.5 on the pairs. (On real data the AUC is optimistic: the cue words were picked on the same training set.)

**Length:** mean words per class and the AUC of length alone. Generated sarcastic sides are longer than serious sides; if length predicts the label much better in the pairs than in real data, a pair-trained model could learn "long = sarcastic". Note it for the paper either way.

In [ ]:
cues = pc.cue_predictiveness(train, pairs, cue_words)
lengths = pc.length_table(train, pairs)
cues.to_csv(OUT_DIR / "cue_predictiveness.csv", index=False)
lengths.to_csv(OUT_DIR / "length.csv", index=False)
display(cues.round(4)); display(lengths.round(3))
print("tables written to", OUT_DIR)